# NL2SQL Training (Kaggle T4)

Thin orchestration notebook. All real code lives in the GitHub repo under `src/`.

**Setup before running:**
1. Settings (right sidebar) → Accelerator → **GPU T4 x2** (or x1, both work)
2. Settings → Internet → **On**
3. Settings → Add-ons → Secrets — add `HF_TOKEN`, `WANDB_API_KEY`, `HF_HUB_REPO`
4. Add Spider + BIRD as Kaggle Datasets to this notebook (Add Input)

In [ ]:
# Cell 1: secrets + clone repo
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
os.environ['HF_HUB_REPO'] = secrets.get_secret('HF_HUB_REPO')

GIT_URL = 'https://github.com/Akashkar00/nl2sql-agent.git'

!cd /kaggle/working && git clone $GIT_URL || (cd /kaggle/working/nl2sql-agent && git pull)
%cd /kaggle/working/nl2sql-agent

In [ ]:
# Cell 2: install deps. Unsloth requires special install order on Kaggle.
!pip install -q --no-deps 'unsloth @ git+https://github.com/unslothai/unsloth.git'
!pip install -q --no-deps 'unsloth_zoo @ git+https://github.com/unslothai/unsloth_zoo.git'
!pip install -q -r requirements.txt
# Note: torch is pre-installed on Kaggle. Don't reinstall — wastes 5min.

In [ ]:
# Cell 3: link Kaggle dataset paths into expected locations
# Adjust these paths based on what you named your Kaggle datasets
!mkdir -p data/spider data/bird
!ln -sfn /kaggle/input/spider-dataset/* data/spider/
!ln -sfn /kaggle/input/bird-dataset/* data/bird/
!ls data/spider/
!ls data/bird/

In [ ]:
# Cell 4: prepare dataset (run once per dataset version)
!python -m src.data.prepare_dataset \
    --spider_root data/spider \
    --bird_root data/bird \
    --output_dir data/processed
!wc -l data/processed/*.jsonl

In [ ]:
# Cell 5: login to HF + W&B
from huggingface_hub import login as hf_login
import wandb
hf_login(token=os.environ['HF_TOKEN'])
wandb.login(key=os.environ['WANDB_API_KEY'])

In [ ]:
# Cell 6: TRAIN. ~6-8 hours on T4 for 2 epochs / 16K examples.
# Adapter pushed to HF Hub every save_steps — survives Kaggle session death.
!python -m src.train --config configs/training_config.yaml

In [ ]:
# Cell 7: quick eval on holdout (sanity check before full benchmark)
!python -m src.eval.run_benchmark \
    --eval_path data/processed/eval_holdout.jsonl \
    --model_name Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path $HF_HUB_REPO \
    --run_name finetuned_with_agent \
    --limit 200